In [20]:
import io
import zipfile
from pathlib import Path
from urllib.parse import quote

import httpx
from tqdm.auto import tqdm
from minsearch import AppendableIndex, Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS
from gitsource import chunk_documents
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.chat.runners import AnthropicMessagesRunner
from toyaikit.llm import AnthropicClient
from toyaikit.tools import Tools

In [19]:
PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

INSTRUCTIONS = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2-3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Pulumi, so you do not need to include the word "pulumi" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [3]:
def get_or_download_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
) -> bytes:
    """
    Return the Markdown documentation as a zip archive, reusing a local copy if one already exists, and downloading it otherwise.

    Args:
        repo_owner (str): The owner of the GitHub repository.
        repo_name (str): The name of the GitHub repository.
        branch_name (str): The branch to download from.
        docs_path (str): Path of the documentation folder inside the repository.
        save_dir (str | None): Local directory to reuse/save the files in. If
            None, the files are only returned as an in-memory zip.

    Returns:
        bytes: A zip archive containing only the Markdown files under `docs_path`.
    """

    def _get(doc_dir: str | None = None) -> bytes | None:
        """Build a zip from an already-downloaded local directory.

        Args:
            doc_dir (str | None): Directory that may already contain the files.

        Returns:
            bytes | None: A zip archive of the local files, or None if the
            directory is missing or empty.
        """
        if doc_dir is None:
            return None

        out_dir = Path(doc_dir)
        existing_files = sorted(out_dir.rglob("*")) if out_dir.exists() else []
        existing_files = [f for f in existing_files if f.is_file()]
        if not existing_files:
            return None

        print(f"Using already-downloaded documentation in '{doc_dir}' ({len(existing_files)} files)")

        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w") as archive:
            for file in existing_files:
                archive.write(file, file.relative_to(out_dir).as_posix())
        return buffer.getvalue()

    def _download() -> bytes:
        """Download only the Markdown documentation from GitHub.

        Returns:
            bytes: A zip archive containing only the Markdown files under
            `docs_path`.
        """
        api = "https://api.github.com"
        raw = "https://raw.githubusercontent.com"

        with httpx.Client(follow_redirects=True) as client:
            # Resolve the branch to its root tree SHA
            branch = client.get(f"{api}/repos/{repo_owner}/{repo_name}/branches/{branch_name}")
            branch.raise_for_status()
            root_tree_sha = branch.json()["commit"]["commit"]["tree"]["sha"]
            print(f"Retrieved root tree SHA: {root_tree_sha}")

            # Locate the docs subtree inside the root tree
            root_tree = client.get(f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{root_tree_sha}")
            root_tree.raise_for_status()
            docs_entry = next(
                (item for item in root_tree.json()["tree"] if item["path"] == docs_path),
                None,
            )
            if docs_entry is None:
                raise ValueError(f"Path '{docs_path}' not found in {repo_owner}/{repo_name}@{branch_name}")
            print(f"Retrieved docs subtree '{docs_path}' (SHA: {docs_entry['sha']})")

            # Recursively list every file under docs/
            docs_tree = client.get(
                f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{docs_entry['sha']}?recursive=1"
            )
            docs_tree.raise_for_status()
            print(f"Retrieved recursive tree for '{docs_path}'")

            # Collect the Markdown files to download
            markdown_files = [
                item["path"]
                for item in docs_tree.json().get("tree", [])
                if item.get("type") == "blob" and item["path"].endswith((".md", ".mdx"))
            ]
            print(f"Found {len(markdown_files)} Markdown files to download")

            # Download each Markdown file and package it into an in-memory zip
            buffer = io.BytesIO()
            with zipfile.ZipFile(buffer, "w") as archive:
                for path in tqdm(markdown_files, desc="Downloading docs", unit="file"):
                    file_url = f"{raw}/{repo_owner}/{repo_name}/{branch_name}/{quote(f'{docs_path}/{path}')}"
                    response = client.get(file_url)
                    response.raise_for_status()
                    archive.writestr(path, response.content)

        return buffer.getvalue()

    # Reuse the local copy if it already exists
    data = _get(save_dir)
    if data is not None:
        return data

    # Otherwise download and (optionally) save to disk
    data = _download()

    if save_dir is not None:
        out_dir = Path(save_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            num_files = len(zf.namelist())
            zf.extractall(out_dir)
        print(f"Extracted {num_files} files to '{save_dir}'")

    return data

In [4]:
def retrieve_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
):
    response_contents = get_or_download_documentation(
        repo_owner=repo_owner,
        repo_name=repo_name,
        branch_name=branch_name,
        docs_path=docs_path,
        save_dir=save_dir,
    )
    documents = []
    with zipfile.ZipFile(io.BytesIO(response_contents)) as zip_ref:
        for file_path in zip_ref.namelist():
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                doc = {
                    'content': content,
                    'filename': file_path
                }
                documents.append(doc)
    return documents

In [5]:
documents = retrieve_documentation(
    repo_owner="pulumi",
    repo_name="docs",
    branch_name="master",
    docs_path="content",
    save_dir="docs",
)
documents[:5]

Using already-downloaded documentation in 'docs' (2186 files)


[{'content': '---\ntitle: The infrastructure as code platform for the AI era – Pulumi\nmeta_desc: Pulumi is an open-source infrastructure as code platform that helps humans and agents build and manage cloud infrastructure with real programming languages.\nmeta_image: /images/home/home-meta.png\n\ninclude_organization_schema: true\n\nsections:\n  - type: hero\n    layout: split\n    cta_secondary_text: Download open source\n    cta_secondary_link: /docs/install/\n    badge_highlight_text: "New:"\n    badge_text: "Native support for Terraform and HCL"\n    badge_link: /releases/terraform-state-backend-modules-hcl/\n    title: "*Unleash agents* <br>on your infrastructure."\n    description: |\n      Ship infrastructure faster with tools that let agents do what they do best — without letting them go off the rails.\n    anchor: hero\n    hero_animation: agent-loop\n\n  - type: logo_grid\n    title: Trusted by over 4,000 innovative companies\n    logos:\n      - snowflake\n      - nvidia\n  

In [6]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [7]:
index = AppendableIndex(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(document_chunks)

In [8]:
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"pulumi"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords),
)

In [21]:
file_index = {doc["filename"]: doc["content"] for doc in documents}
file_index["blog/10-things-you-can-do-with-neo/index.md"]

'---\ntitle: "10 Things You Can Do With Our Infrastructure Agent, Neo"\nfeature_image: feature.png\ndate: 2025-10-06\ndraft: false\nmeta_desc: "Discover 10 concrete ways Pulumi Neo helps automate infrastructure tasks while keeping humans in control."\nauthors:\n    - meagan-cojocar\ntags:\n    - ai\n    - platform-engineering\n    - pulumi-neo\n    - aws\nseries: neo-things\ncategory: product\n\nsocial:\n    twitter: "10 concrete workflows you can use Pulumi Neo, our new infrastructure agent, for: auto-fix AWS config violations, upgrade Lambda runtimes across accounts, generate infrastructure templates, respond to CVEs. Neo generates PRs, not production changes."\n    linkedin: "10 concrete workflows platform teams can use Pulumi Neo, our new infrastructure agent, for right now. From auto-remediating AWS config violations to upgrading Lambda runtimes across multiple accounts, Neo handles tasks that used to take days. Neo generates PRs, not one-way tickets. You stay in control while AI 

In [16]:
class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self, index: AppendableIndex, highlighter: Highlighter, file_index: dict[str, str],
    ) -> None:
        """
        Initialize the SearchTools instance.
        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (dict[str, str]): Mapping of filenames to file contents.
        """

        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> list[dict]:
        """
        Search the index for results matching a query and highlight them.
        Args:
            query (str): The search query to look up in the index.
        Returns:
            list[dict[str, Any]]: A list of highlighted search result objects.
        """

        search_results = self.index.search(query=query, num_results=5)
        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.
        Args:
            filename (str): The filename of the file to retrieve.
        Returns:
            str: The file contents if found, otherwise an error message.
        """

        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

In [17]:
search_tools = SearchTools(index, highlighter, file_index)

In [22]:
agent_tools = Tools()
agent_tools.add_tools(search_tools)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'get_file',
  'description': "Retrieve a file's contents by filename.\nArgs:\n    filename (str): The filename of the file to retrieve.\nReturns:\n    str: The file contents if found, otherwise an error message.",
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'}},
   'required': ['filename'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the index for results matching a query and highlight them.\nArgs:\n    query (str): The search query to look up in the index.\nReturns:\n    list[dict[str, Any]]: A list of highlighted search result objects.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
llm_client = AnthropicClient(model="claude-opus-5")
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)  # type: ignore
agent = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,  # type: ignore
    llm_client=llm_client,
)

In [24]:
result = agent.loop(
    "How do I launch an EC2 in a private net in AWS?",
    callback=runner_callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


In [17]:
result2 = agent.loop(
    "Show me how to do that in python",
    callback=runner_callback,
    previous_messages=result.all_messages,
)

-> Response received


In [25]:
float(result.cost.total_cost) if result.cost else 0.00

0.23016